# Karimi Proposal

In [1]:
#DO NOT CHANGE THIS CODE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pmtools2 as pm
import kmodes
%matplotlib inline


## URL del código.
url_dataset = "/Users/ddiaz/Documents/code/phd-thesis-lab/12-third_year/00-Data/02-HC/00-Original/0-HC-universal.csv"

In [11]:
# LOAD DATA
#DO NOT CHANGE THIS CODE
df_emp_access = pd.read_csv(url_dataset)
df_emp_access = df_emp_access[df_emp_access.columns[1:]]
attnames_list = list(df_emp_access.columns)
print(attnames_list)
#A_III Statistics

#DO NOT CHANGE THIS CODE
entries = df_emp_access.values.tolist()
attidx_to_collcounter = pm.get_entries_freqs(entries)

USER_ATTRS = ['position', 'specialties', 'teams', 'uward', 'agentfor']
RESOURCE_ATTRS = ['type', 'patient', 'tratingTeam',
                    'rward', 'author', 'topics']

df_emp_access = df_emp_access.rename(columns={'rname': 'RESOURCE'})


print('#Entries:', len(entries))
print('#Users:', len(df_emp_access.uname.drop_duplicates()))
print('#Resources:', len(df_emp_access[RESOURCE_ATTRS].drop_duplicates()))
print()

['uname', 'position', 'uward', 'specialties', 'teams', 'agentfor', 'type', 'author', 'patient', 'topics', 'tratingTeam', 'rward', 'rname', 'ACTION']
#Entries: 153760
#Users: 31
#Resources: 4960



In [12]:
print('---Attributes---')
for att_idx in range(len(attnames_list)):
    print('Num values', attnames_list[att_idx], ':', len(attidx_to_collcounter[att_idx].keys()))
num_resources = attnames_list[1]

#B_I Filter resources

#DO NOT CHANGE THIS CODE
n1 = 0
n2 = 100
num_resources = len(df_emp_access[RESOURCE_ATTRS].drop_duplicates())
top_list = df_emp_access['RESOURCE'].value_counts()[:num_resources].index.tolist()
#Filter the interval between n1 and n2
top_list = top_list[n1:n2+1]
print('#Filtered resources:', len(top_list))
    
print()
df_pos_entries = df_emp_access[df_emp_access['ACTION']==1]
df_neg_entries = df_emp_access[df_emp_access['ACTION']==0]
print('Num positive entries:',len(df_pos_entries))
print('Num negative entries:',len(df_neg_entries))

#DO NOT CHANGE THIS CODE
boolean_series = df_pos_entries.RESOURCE.isin(top_list)
df_pos_entries_ = df_pos_entries[boolean_series]
boolean_series = df_neg_entries.RESOURCE.isin(top_list)
df_neg_entries_ = df_neg_entries[boolean_series]

df_pos_entries = df_pos_entries_
df_neg_entries = df_neg_entries_

pos_entries = df_pos_entries_.values.tolist()
neg_entries = df_neg_entries_.values.tolist()

print('#Resources:', len(df_pos_entries.RESOURCE.drop_duplicates()))

---Attributes---
Num values uname : 31
Num values position : 3
Num values uward : 3
Num values specialties : 6
Num values teams : 5
Num values agentfor : 3
Num values type : 2
Num values author : 31
Num values patient : 2
Num values topics : 5
Num values tratingTeam : 4
Num values rward : 2
Num values rname : 4960
Num values ACTION : 2
#Filtered resources: 101

Num positive entries: 24720
Num negative entries: 129040
#Resources: 101


## Data pre-processing
1. Continouos to Categorical Values
2. Handle missing values - New value

In [13]:
# df_data = df_pos_entries[attnames_list[2:]].drop_duplicates()
# print('Drop duplicates:', len(df_data))

# pos_entries = df_data.values.tolist()
pos_entries = df_pos_entries.values.tolist()
neg_entries = df_neg_entries.values.tolist()

## Selection of Learning Algorithm
1. K-modes algorithm

In [14]:
###Select the number of clusters###
num_clusters = 15

#DO NOT CHANGE THIS CODE
# seed = 29

#Compute centroids and labels
# num_init = 5
centroids = []
kmodes_huang = kmodes.KModes(n_clusters=num_clusters, init='Huang', verbose=0)
cluster_labels = kmodes_huang.fit_predict(df_pos_entries)
centroids = kmodes_huang.cluster_centroids_

print('Ready!')    

Ready!


In [15]:
df_pos_entries["cls"] = cluster_labels

## Parameter tuning
1. Number of clusters (Silhouette Method)
2. Cluster initialization

## Policy Rules Extraction

In [16]:
def freq(value, attribute, dataplace):
    """
    Calculate the frequency of the value in the dataplace.

    Parameters
    ----------
    value : int
        Value to compute its frequency.

    attribute : string
        Name of the attribute.

    dataplace : DataFrame pandas
        Data to search.
    
    Returns
    -------
    float [0-1]
        Returns the value of frequency of the value in the data.
    """
    value_freq = dataplace[dataplace[attribute] == value].drop_duplicates()
    return len(value_freq) / len(dataplace)

def freq_rels(attrA, attrB, dataplace):
    """
    Compute the frequency of the attribute relation.

    Parameters
    ----------
    attrA : string
        Name of the attribute A to compare.
    attrB : string
        Name of the attribute B to compare.
    dataplace : DataFrame pandas
        Data to search.

    Returns
    -------
    float [0-1]
        Returns the value of frequency of the range of values in the data.
    """
    # Get the range of values of attribute A.
    range_val_A = set(dataplace[attrA].values.tolist())

    # Get the range of values of attribute B.
    range_val_B = set(dataplace[attrB].values.tolist())

    # Check if the len
    if len(range_val_A) == len(range_val_B):
        # Compute the intersection
        inter_A_B = range_val_A.intersection(range_val_B)
        if len(inter_A_B) == len(range_val_A):
            boolean_series = dataplace[attrA].isin(inter_A_B)
            frac_log = dataplace[boolean_series]
            return len(frac_log) / len(dataplace) # Return the fraction
        return 0
    else:
        return 0    

def extract_attributes_filters(C_i, A, L, posThr, negThr):
    """
    Effective attribute extraction algorithm. Generate a rule for each cluster.

    Parameters
    ----------
    C_i : DataFrame pandas
        Access request in the Cluster i.

    A : List
        List of attributes.

    V : List
        Values of attributes.

    L : DataFrame
        Complete Access Log.

    PosThr : float
        Positive Threshold to the effective positive attribute.

    NegThr : float
        Negative Threshold to the effective negative attribute.

    Returns
    -------
    list
        Returns the rule with the effective attributes for the cluster i.
    """
    filter_to_ret = [] # Rule
    for a in A:        
        a_values = C_i[a].drop_duplicates().tolist()        
        for v in a_values:
            if freq(v, a, C_i) - freq(v, a, L) > posThr:
                if not [a, v] in filter_to_ret:
                    filter_to_ret.append([a, v])
            if freq(v, a, L) - freq(v, a, C_i) > negThr:
                if not [a, -1*v] in filter_to_ret:
                    filter_to_ret.append([a, v*-1])
    return filter_to_ret

def extract_relations(C_i, A, L, posThr, negThr):
    """
    Extract the effective relation. For each cluster.

    Parameters
    ----------
    C_i : DataFrame pandas
        Access request in the Cluster i.

    A : List
        List of attributes.

    L : DataFrame
        Complete Access Log.

    posThr : float
        Positive Threshold to the effective positive relation.

    negThr : float
        Negative Threshold to the effective negative relation.

    Returns
    -------
    list
        Returns the rule with the effective relation for the cluster i.
    """
    relation_to_ret = []
    for a in A:
        for b in A:
            if a != b:
                if freq_rels(a, b, C_i) - freq_rels(a, b, L) > posThr:
                    if not [a, b] in relation_to_ret:
                        relation_to_ret.append([a, b])
                if freq_rels(a, b, L) - freq_rels(a, b, C_i) > negThr:
                    if not [a, '!'+b] in relation_to_ret:                        
                        relation_to_ret.append([a, '!'+b])
                    #print()

def rule_inference(data_, pos_attr_thr, 
    neg_attr_thr, pos_rel_thr, neg_rel_thr):
    rule_list = [] # All rules
    n_cluster = len(data_["cls"].drop_duplicates()) # N clusters
    attrs = data_.columns[:-1] # Name of the columns

    for C_i in range(n_cluster):
        #print(C_i)
        rule_i = []
        data_cluster = data_[data_["cls"] == C_i]
        
        # Effective attributes
        attr_filters = extract_attributes_filters(data_cluster, attrs, data_, 
            pos_attr_thr, neg_attr_thr)    
        rule_i.append(attr_filters)        

        # Relations
        attr_relation = extract_relations(data_cluster, attrs, data_, 
            pos_rel_thr, neg_rel_thr)
        rule_i.append(attr_relation)
        #print(rule_i)

        rule_list.append([C_i, rule_i])    

    return rule_list

In [17]:
df_test = df_pos_entries[df_pos_entries.columns[1:]]
df_test.columns

Index(['position', 'uward', 'specialties', 'teams', 'agentfor', 'type',
       'author', 'patient', 'topics', 'tratingTeam', 'rward', 'RESOURCE',
       'ACTION', 'cls'],
      dtype='object')

In [18]:
pos_attr_thr = 0.3
neg_attr_thr = 0.2
pos_rel_thr = 0.3
neg_rel_thr = 0.2
test = rule_inference(df_test, pos_attr_thr, neg_attr_thr, pos_rel_thr, neg_attr_thr)
len(test)

15

In [19]:
only_rules = []
for rule in test:
    only_rules.append(rule[1][0])
len(only_rules)

15

In [20]:
# Compute how many rules has a lenght equal to 1.
for idx, rule in enumerate(only_rules):
    if len(rule) < 2:
        print(idx)

In [58]:
del only_rules[12]

In [21]:
false_neg  = []
for i,row in df_pos_entries.iterrows():
    
    # Evaluación
    denies_count = 0    
    for rule in only_rules:                                      
        # En esta parte se evalua la regla completa
        res = True
        
        #for idx_r, attr_val in enumerate(rule):
        for attr_val in rule:            
            if attr_val[1] < 0:
                if row[attr_val[0]] == attr_val[1]*-1:
                    res = False
                    break
            else:
                if row[attr_val[0]] != attr_val[1]:
                    res = False
                    break
        if res == False:
            denies_count += 1
    
    if denies_count == len(only_rules):
        false_neg.append(row)
        #print("FP-2")

TypeError: '<' not supported between instances of 'str' and 'int'

In [61]:
FN = len(false_neg)
print("Tasa FN: {:.2f}".format((FN/ len(df_pos_entries))*100))
print("FN: ", FN, " de ", len(df_pos_entries))

Tasa FN: 34.82
FN:  10749  de  30872


In [62]:
false_pos  = []
for i,row in df_neg_entries.iterrows():
    # Evaluación
    denies_count = 0
    temp_rules_n = 0
    for rule in only_rules:                                      
        # En esta parte se evalua la regla completa
        res = True                        
        for attr_val in rule:
            if attr_val[1] < 0:
                if row[attr_val[0]] == attr_val[1]*-1:
                    res = False
                    break
            else:
                if row[attr_val[0]] != attr_val[1]:
                    res = False
                    break
        if res == False:
            denies_count += 1                                
    #print("XXX-", denies_count, temp_rules_n, res)
    if denies_count < len(only_rules):
        false_pos.append(row)
        #print("FP-2")    
    #else:
    #    print("ENtra PAPA")
    

In [64]:
FP = len(false_pos)
print("Tasa FP: {:.2f}".format((FP/ len(df_neg_entries))*100))
print("FP: ", FP, " de ", len(df_neg_entries))

Tasa FP: 62.68
FP:  1189  de  1897


In [65]:

TP = len(df_pos_entries) - FN
#TP = 50 - FN
TN = len(df_neg_entries) - FP
#TN = 50 - FP

precision = TP / (TP + FP)

recall = TP / (TP + FN)

fscore = 2*(precision*recall)/(precision+recall)

print("FN:", FN, " - {:.2f}".format((FN/len(df_pos_entries))*100))
#print("FN:", FN, " - {:.2f}".format((FN/50)*100))
print("FP:", FP, " - {:.2f}".format((FP/len(df_neg_entries))*100))
#print("FP:", FP, " - {:.2f}".format((FP/50)*100))
print("Precision:", precision)
print("Recall:", recall)
print("F-score", fscore)

def compute_wsc(policy):
    return sum([len(rule) for rule in policy])

print("# Rules:", len(only_rules))
print("WSC:", compute_wsc(only_rules))

FN: 10749  - 34.82
FP: 1189  - 62.68
Precision: 0.9442098348348348
Recall: 0.6518204197978751
F-score 0.7712325617047371
# Rules: 13
WSC: 51


## Policy Enhancement
1. Rule Pruning
2. Policy Refinement

In [18]:
def convert_rule_to_str(rule):
    rule_str = []
    #print(rule)
    for item in rule:
        rule_str.append(item[0]+'*'+str(item[1]))
    return set(rule_str)

def convert_set_to_list(rule):
    rule_list = []    
    for item in rule:
        rule_list.append([item.split('*')[0], int(item.split('*')[1])])
    return rule_list

def jaccard_similarity(rule_i, rule_j):

    #transofrm data
    rule_i_str = convert_rule_to_str(rule_i)
    rule_j_str = convert_rule_to_str(rule_j)

    intersection = len(list(set(rule_i_str).intersection(rule_j_str)))
    union = len(list(set(rule_i_str).union(rule_j_str)))
    return float( intersection / union ) 

def get_similar_rules(rule_i, all_rules):
    similar_rules = []
    for rule_j in all_rules:
        # Jaccard similarity
        jaccard_sim = jaccard_similarity(rule_i, rule_j)
        if jaccard_sim > 0.5:
            similar_rules.append(rule_j)

    return similar_rules

def fn_refine_policy(fn_rules, all_rules):
    new_rules = all_rules
    for rule in fn_rules:
        
        similar_rules = get_similar_rules(rule, all_rules)

        if len(similar_rules) == 0:
            new_rules.append(rule)
        else:
            for sim_rule in similar_rules:
                rule_str = convert_rule_to_str(rule)
                rule_str_sim = convert_rule_to_str(sim_rule)

                new_filter = rule_str_sim.difference((rule_str_sim.difference(rule_str)))
                print(new_filter)
                new_list_filter = convert_set_to_list(new_filter)
                idx_to_del = new_rules.index(sim_rule)
                del new_rules[idx_to_del]
                new_rules.append(new_list_filter)
    
    return new_rules
                

def fp_refine_policy(fp_rules, all_rules):
    new_rules = all_rules
    for rule in fp_rules:
        
        similar_rules = get_similar_rules(rule, all_rules)

        if len(similar_rules) != 0:                    
            for sim_rule in similar_rules:
                rule_str = convert_rule_to_str(rule)
                rule_str_sim = convert_rule_to_str(sim_rule)

                new_filter = rule_str_sim.difference((rule_str_sim.difference(rule_str)))
                new_list_filter = convert_set_to_list(new_filter)
                idx_to_del = new_rules.index(sim_rule)
                del new_rules[idx_to_del]
                new_rules.append(new_list_filter)
    
    return new_rules

In [66]:
false_neg = pd.DataFrame(false_neg)

In [67]:
###Select the number of clusters###
num_clusters = 5

#DO NOT CHANGE THIS CODE
# seed = 29

#Compute centroids and labels
# num_init = 5
centroids = []
kmodes_huang = kmodes.KModes(n_clusters=num_clusters, init='Huang', verbose=0)
cluster_labels = kmodes_huang.fit_predict(false_neg)
centroids = kmodes_huang.cluster_centroids_

print('Ready!')    

Ready!


In [68]:
false_neg["cls"] = cluster_labels

In [69]:
df_test_2 = false_neg[false_neg.columns[1:]]

In [70]:
pos_attr_thr = 0.3
neg_attr_thr = 0.2
pos_rel_thr = 0.3
neg_rel_thr = 0.2
test = rule_inference(df_test_2, pos_attr_thr, neg_attr_thr, pos_rel_thr, neg_attr_thr)
len(test)

5

In [71]:
only_rules_2 = []
for rule in test:
    only_rules_2.append(rule[1][0])
len(only_rules_2)

5

In [72]:
only_rules_2 = []
for rule in test:
    only_rules_2.append(rule[1][0])
len(only_rules_2)

5

In [73]:
# Compute how many rules has a lenght equal to 1.
for idx, rule in enumerate(only_rules):
    if len(rule) < 2:
        print(idx)

In [29]:
del only_rules[4]

In [74]:
new_rules = fn_refine_policy(only_rules_2, only_rules)
len(new_rules)

18

In [75]:
false_neg  = []
for i,row in df_pos_entries.iterrows():
    
    # Evaluación
    denies_count = 0    
    for rule in only_rules:                                      
        # En esta parte se evalua la regla completa
        res = True
        
        #for idx_r, attr_val in enumerate(rule):
        for attr_val in rule:            
            if attr_val[1] < 0:
                if row[attr_val[0]] == attr_val[1]*-1:
                    res = False
                    break
            else:
                if row[attr_val[0]] != attr_val[1]:
                    res = False
                    break
        if res == False:
            denies_count += 1
    
    if denies_count == len(only_rules):
        false_neg.append(row)
        #print("FP-2")

FN = len(false_neg)
print("Tasa FN: {:.2f}".format((FN/ len(df_pos_entries))*100))
print("FN: ", FN, " de ", len(df_pos_entries))

Tasa FN: 20.70
FN:  6389  de  30872


In [76]:
false_pos  = []
for i,row in df_neg_entries.iterrows():
    # Evaluación
    denies_count = 0
    temp_rules_n = 0
    for rule in only_rules:                                      
        # En esta parte se evalua la regla completa
        res = True                        
        for attr_val in rule:
            if attr_val[1] < 0:
                if row[attr_val[0]] == attr_val[1]*-1:
                    res = False
                    break
            else:
                if row[attr_val[0]] != attr_val[1]:
                    res = False
                    break
        if res == False:
            denies_count += 1                                
    #print("XXX-", denies_count, temp_rules_n, res)
    if denies_count < len(only_rules):
        false_pos.append(row)
        #print("FP-2")    
    #else:
    #    print("ENtra PAPA")
FP = len(false_pos)
print("Tasa FP: {:.2f}".format((FP/ len(df_neg_entries))*100))
print("FN: ", FP, " de ", len(df_neg_entries))

Tasa FP: 79.28
FN:  1504  de  1897


In [82]:
TP = len(df_pos_entries) - FN
#TP = 50 - FN
TN = len(df_neg_entries) - FP
#TN = 50 - FP

precision = TP / (TP + FP)

recall = TP / (TP + FN)

fscore = 2*(precision*recall)/(precision+recall)

print("FN:", FN, " - {:.2f}".format((FN/len(df_pos_entries))*100))
#print("FN:", FN, " - {:.2f}".format((FN/50)*100))
print("FP:", FP, " - {:.2f}".format((FP/len(df_neg_entries))*100))
#print("FP:", FP, " - {:.2f}".format((FP/50)*100))
print("Precision:", precision)
print("Recall:", recall)
print("F-score", fscore)

def compute_wsc(policy):
    return sum([len(rule) for rule in policy])

print("# Rules:", len(only_rules))
print("WSC:", compute_wsc(only_rules))

FN: 6389  - 20.70
FP: 1504  - 79.28
Precision: 0.9421249086081502
Recall: 0.7930487172842705
F-score 0.8611829261858283
# Rules: 18
WSC: 65


In [86]:
for r in only_rules:
    print(r)
    

[['ROLE_ROLLUP_1', -117961], ['ROLE_FAMILY_DESC', -117906], ['ROLE_FAMILY', -290919]]
[['ROLE_ROLLUP_2', 118327], ['ROLE_TITLE', 118321], ['ROLE_FAMILY', 290919], ['ROLE_CODE', 118322]]
[['ROLE_ROLLUP_2', 118343], ['ROLE_FAMILY', 118638], ['ROLE_FAMILY', -290919]]
[['ROLE_ROLLUP_2', 118343], ['ROLE_FAMILY', 118424]]
[['ROLE_ROLLUP_1', 117961], ['ROLE_ROLLUP_2', 118300]]
[['ROLE_ROLLUP_2', 118300], ['ROLE_TITLE', 120344], ['ROLE_FAMILY', 118424], ['ROLE_CODE', 120346]]
[['ROLE_ROLLUP_1', 117961], ['ROLE_ROLLUP_2', 118225]]
[['ROLE_ROLLUP_1', 91261], ['ROLE_ROLLUP_1', -117961], ['ROLE_ROLLUP_2', 118026], ['ROLE_TITLE', 117905], ['ROLE_FAMILY', 290919], ['ROLE_CODE', 117908]]
[['ROLE_ROLLUP_1', 117961], ['ROLE_TITLE', 117905], ['ROLE_FAMILY_DESC', 117906], ['ROLE_FAMILY', 290919], ['ROLE_CODE', 117908]]
[['ROLE_ROLLUP_1', 117961], ['ROLE_ROLLUP_2', 117962]]
[['ROLE_ROLLUP_1', 117961], ['ROLE_ROLLUP_2', 118327], ['ROLE_DEPTNAME', 118391], ['ROLE_TITLE', 117905], ['ROLE_FAMILY_DESC', 117906

In [79]:
###Select the number of clusters###
num_clusters = 5

#DO NOT CHANGE THIS CODE
# seed = 29

#Compute centroids and labels
# num_init = 5
centroids = []
kmodes_huang = kmodes.KModes(n_clusters=num_clusters, init='Huang', verbose=0)
cluster_labels = kmodes_huang.fit_predict(false_neg)
centroids = kmodes_huang.cluster_centroids_

print('Ready!')    

Ready!


In [81]:
false_neg["cls"] = cluster_labels

TypeError: list indices must be integers or slices, not str

In [87]:
df_test_2 = false_neg[false_neg.columns[1:]]

In [88]:
pos_attr_thr = 0.3
neg_attr_thr = 0.2
pos_rel_thr = 0.3
neg_rel_thr = 0.2
test = rule_inference(df_test_2, pos_attr_thr, neg_attr_thr, pos_rel_thr, neg_attr_thr)
len(test)

5

In [89]:
#only_rules = []
for rule in test:
    only_rules.append(rule[1][0])
len(only_rules)

24

In [91]:
# Compute how many rules has a lenght equal to 1.
for idx, rule in enumerate(only_rules):
    if len(rule) == 1:
        print(idx)

In [80]:
del only_rules[16]

In [92]:
false_neg  = []
for i,row in df_pos_entries.iterrows():
    
    # Evaluación
    denies_count = 0    
    for rule in only_rules:                                      
        # En esta parte se evalua la regla completa
        res = True
        
        #for idx_r, attr_val in enumerate(rule):
        for attr_val in rule:            
            if attr_val[1] < 0:
                if row[attr_val[0]] == attr_val[1]*-1:
                    res = False
                    break
            else:
                if row[attr_val[0]] != attr_val[1]:
                    res = False
                    break
        if res == False:
            denies_count += 1
    
    if denies_count == len(only_rules):
        false_neg.append(row)
        #print("FP-2")

FN = len(false_neg)
print("Tasa FN: {:.2f}".format((FN/ len(df_pos_entries))*100))
print("FN: ", FN, " de ", len(df_pos_entries))

Tasa FN: 0.00
FN:  0  de  30872


In [93]:
false_pos  = []
for i,row in df_neg_entries.iterrows():
    # Evaluación
    denies_count = 0
    temp_rules_n = 0
    for rule in only_rules:                                      
        # En esta parte se evalua la regla completa
        res = True                        
        for attr_val in rule:
            if attr_val[1] < 0:
                if row[attr_val[0]] == attr_val[1]*-1:
                    res = False
                    break
            else:
                if row[attr_val[0]] != attr_val[1]:
                    res = False
                    break
        if res == False:
            denies_count += 1                                
    #print("XXX-", denies_count, temp_rules_n, res)
    if denies_count < len(only_rules):
        false_pos.append(row)
        #print("FP-2")    
    #else:
    #    print("ENtra PAPA")
FP = len(false_pos)
print("Tasa FP: {:.2f}".format((FP/ len(df_neg_entries))*100))
print("FN: ", FP, " de ", len(df_neg_entries))

Tasa FP: 100.00
FN:  1897  de  1897


In [94]:
TP = len(df_pos_entries) - FN
#TP = 50 - FN
TN = len(df_neg_entries) - FP
#TN = 50 - FP

precision = TP / (TP + FP)

recall = TP / (TP + FN)

fscore = 2*(precision*recall)/(precision+recall)

print("FN:", FN, " - {:.2f}".format((FN/len(df_pos_entries))*100))
#print("FN:", FN, " - {:.2f}".format((FN/50)*100))
print("FP:", FP, " - {:.2f}".format((FP/len(df_neg_entries))*100))
#print("FP:", FP, " - {:.2f}".format((FP/50)*100))
print("Precision:", precision)
print("Recall:", recall)
print("F-score", fscore)

def compute_wsc(policy):
    return sum([len(rule) for rule in policy])

print("# Rules:", len(only_rules))
print("WSC:", compute_wsc(only_rules))

FN: 0  - 0.00
FP: 1897  - 100.00
Precision: 0.9421099209618847
Recall: 1.0
F-score 0.9701921717132037
# Rules: 24
WSC: 92
